# 💻 GUÍA PRÁCTICA AVANZADA: ARQUITECTURA MODEL CONTEXT PROTOCOL (MCP)
## Visualizando Paquetes JSON-RPC, Servidores Web SSE y Swapping de Clientes/SDKs (Nvidia Nemotron vs Google Gemini)

En esta versión corregida y adaptada para clústeres de Databricks, implementaremos la arquitectura MCP con los siguientes estándares:
1. **Eliminación del Error de 'uv'**: Usamos `sys.executable` para levantar el Servidor MCP en segundo plano, garantizando compatibilidad absoluta con Databricks sin requerir binarios externos de gestión de paquetes.
2. **Swapping de Cerebros y SDKs Reales**: Simplificamos la selección a las dos opciones reales:
   * **Opción 1: Google Gemini** (Usando el SDK oficial `google-generativeai` con tu clave de API gratuita).
   * **Opción 2: Nvidia Nemotron** (Usando el SDK de `openai` a través de OpenRouter con tu clave unificada).
3. **Visualización de Paquetes JSON-RPC (Tuberías en Vivo)**: Dibujaremos tarjetas HTML de diseño premium en la consola que muestren las peticiones y respuestas JSON crudas del estándar MCP.

---

## 📦 Paso 1: Instalación de Dependencias (Específico para Databricks)
Instalamos todas las librerías necesarias directamente en el clúster de Databricks, incluyendo el SDK oficial de Google Generative AI.

In [ ]:
# Comando mágico de Databricks para instalar todas las dependencias del proyecto en el clúster
%pip install mcp openai pandas scikit-learn numpy sqlalchemy pymysql python-dotenv google-generativeai

In [ ]:
# Reiniciar el entorno de ejecución de Python en el clúster para registrar los nuevos paquetes instalados
%restart_python

---

## 🛠️ Paso 2: Importar Librerías y Herramientas Visuales
Importamos los componentes estándar de MCP, OpenAI, Google Generative AI y utilidades de visualización web de Jupyter.

In [ ]:
import os
import sys
import asyncio
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
import google.generativeai as genai
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.sse import sse_client
from IPython.display import display, HTML

---

## 🔑 Paso 3: Carga Segura y Verificación de Credenciales (.env)
Cargamos las variables privadas del archivo local `.env` y validamos que al menos una clave de API esté configurada.

In [ ]:
# Cargar variables del .env
load_dotenv(override=True)

gemini_key = os.getenv("GEMINI_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

print("📋 Estado de las Credenciales Cargadas:")
print(f"   ➔ GEMINI_API_KEY (Google SDK): {'✅ CONFIGURADA' if gemini_key else '❌ NO CONFIGURADA'}")
print(f"   ➔ OPENROUTER_API_KEY (OpenRouter): {'✅ CONFIGURADA' if openrouter_key else '❌ NO CONFIGURADA'}")

---

## 🌐 Paso 4: Selección de Proveedor y Modelo de IA (Google vs OpenRouter)
Elige con qué cerebro de IA procesar la tarea. Al cambiarlo, verás que el comportamiento del Agente cambia y utilizará SDKs totalmente diferentes para orquestar los tools locales en segundo plano.

In [ ]:
print("=======================================================================")
print(" SELECCIÓN DE MODELO E INTEGRACIÓN:")
print("-----------------------------------------------------------------------")
print(" [1] -> google/gemini-1.5-flash (Usa el SDK oficial google-generativeai)")
print(" [2] -> nvidia/nemotron-3-super-120b-a12b:free (Usa el SDK openai con OpenRouter)")
print("=======================================================================\n")

opcion_modelo = input("🧠 Selecciona tu modelo (1 o 2, ENTER para predeterminado [1]): ").strip()
ia_proveedor = "gemini" if opcion_modelo != "2" else "openrouter"
modelo_activo = "gemini-1.5-flash" if ia_proveedor == "gemini" else "nvidia/nemotron-3-super-120b-a12b:free"

# Validación de clave específica según la opción elegida
if ia_proveedor == "gemini" and not gemini_key:
    raise ValueError("❌ Error: Elegiste Google Gemini pero la variable GEMINI_API_KEY está vacía en tu .env.")
elif ia_proveedor == "openrouter" and not openrouter_key:
    raise ValueError("❌ Error: Elegiste Nvidia/OpenRouter pero la variable OPENROUTER_API_KEY está vacía en tu .env.")

print(f"\n✅ Inferencia configurada sobre el proveedor: '{ia_proveedor.upper()}'")
print(f"✅ Modelo activo: '{modelo_activo}'")

---

## 🔌 Paso 5: Selección de Transporte (¿Stdio o SSE?)
Configuramos el canal de comunicación local con el servidor MCP.

**CORRECCIÓN DE ERRORES DE DATABRICKS**: En lugar de usar `uv` (que no existe por defecto en los nodos del clúster de Databricks), usamos `sys.executable` para invocar directamente el intérprete de Python del entorno activo.

In [ ]:
print("=======================================================================")
print(" SELECCIÓN DE TRANSPORTE MCP:")
print("-----------------------------------------------------------------------")
print(" [1] -> stdio (Servidor integrado de consola con Python nativo - RECOMENDADO)")
print(" [2] -> sse   (Servidor Web local por red http://localhost:8000/sse)")
print("=======================================================================\n")

opcion_transporte = input("🔌 Selecciona el transporte (1 o 2, ENTER para predeterminado [1]): ").strip()
transporte_mcp = "sse" if opcion_transporte == "2" else "stdio"

# Configurar los parámetros si usamos stdio (con sys.executable para evitar fallos de 'uv')
server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_server.py"],
    env=os.environ.copy()
)

print(f"\n✅ Transporte MCP configurado en modo: '{transporte_mcp.upper()}'")

---

## 🎨 Paso 6: Definición del Sniffer Visual de Paquetes JSON-RPC
Esta función intercepta las peticiones y dibuja tarjetas interactivas de diseño premium neón en tu Notebook.

In [ ]:
def mostrar_log_visual_json_rpc(direccion, metodo, payload):
    """Dibuja una tarjeta con diseño premium para visualizar el flujo JSON-RPC de MCP."""
    if direccion == "CLIENTE_ENVIANDO":
        titulo = "📤 CLIENTE MCP -> SOLICITANDO ACCIÓN (JSON-RPC OUTBOX)"
        borde_color = "#00c3ff"
        bg_color = "rgba(0, 195, 255, 0.05)"
    elif direccion == "SERVIDOR_RESPONDIENDO":
        titulo = "📥 SERVIDOR MCP -> RETORNANDO RESPUESTA (JSON-RPC INBOX)"
        borde_color = "#00ff88"
        bg_color = "rgba(0, 255, 136, 0.05)"
    else:
        titulo = "🧠 INTELIGENCIA ARTIFICIAL -> DECISIÓN DE INVOCAR HERRAMIENTA"
        borde_color = "#ffaa00"
        bg_color = "rgba(255, 170, 0, 0.05)"
        
    json_bonito = json.dumps(payload, indent=2, ensure_ascii=False)
    
    html_code = f"""
    <div style="
        border: 2px solid {borde_color};
        background-color: {bg_color};
        color: #e2e8f0;
        padding: 15px;
        border-radius: 8px;
        margin: 15px 0;
        font-family: 'Consolas', monospace;
        box-shadow: 0 4px 15px rgba(0, 0, 0, 0.3);
    ">
        <div style="font-weight: bold; color: {borde_color}; margin-bottom: 8px; font-size: 1.1em;">
            {titulo}
        </div>
        <div style="font-size: 0.9em; margin-bottom: 5px; color: #94a3b8;">
            <strong>Método/Acción:</strong> <code>{metodo}</code>
        </div>
        <pre style="
            background-color: #0f172a;
            padding: 10px;
            border-radius: 5px;
            overflow-x: auto;
            border: 1px solid #1e293b;
            color: #38bdf8;
            margin: 0;
        ">{json_bonito}</pre>
    </div>
    """
    display(HTML(html_code))

---

## 💬 Paso 7: Configuración Interactiva de la Consulta (Prompt)
Se abrirá una casilla interactiva. Presiona Enter directamente para usar el prompt predeterminado o escribe tu propio requerimiento.

In [ ]:
prompt_predeterminado = (
    "Por favor, extrae 100 transacciones de la base de datos MySQL "
    "utilizando tus herramientas locales y luego procesa los datos "
    "para dejarlos listos para entrenar un modelo de ML."
)

print("=======================================================================")
print(" SUGERENCIAS PARA EL MODELO DE IA:")
print("-----------------------------------------------------------------------")
print(" [Por defecto] -> (Solo presiona Enter en la casilla)")
print(f"   '{prompt_predeterminado}'\n")
print(" [Sugerencia 1] -> 'Extrae solo 10 transacciones de MySQL y no hagas nada más'")
print("=======================================================================\n")

user_input = input("✍️ Escribe tu instrucción (o presiona ENTER para usar la predeterminada): ").strip()
prompt_activo = user_input if user_input else prompt_predeterminado
print(f"\n✅ Prompt configurado: '{prompt_activo}'")

---

## 🚀 Paso 8: Conexión y Orquestación del Pipeline MCP con Logs JSON-RPC
Esta celda ejecuta la lógica unificada del Cliente MCP. Se conecta mediante el transporte configurado, lee las herramientas locales, orquesta la inferencia y procesa las llamadas de base de datos.

In [ ]:
async def correr_pipeline_mcp_avanzado(prompt, proveedor, modelo):
    # 1. Configuración del Cliente de Inferencia según Proveedor/SDK
    if proveedor == "gemini":
        genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
    else:
        llm_client = AsyncOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY")
        )
        
    # 2. Definición del canal de comunicación del Servidor MCP
    if transporte_mcp == "sse":
        cliente_mcp = sse_client("http://localhost:8000/sse")
    else:
        cliente_mcp = stdio_client(server_params)
        
    print(f"🔌 [MCP] Conectando vía transporte {transporte_mcp.upper()}...")
    
    async with cliente_mcp as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # A. LOG VISUAL: Solicitando herramientas al servidor
            mostrar_log_visual_json_rpc(
                direccion="CLIENTE_ENVIANDO",
                metodo="tools/list",
                payload={"jsonrpc": "2.0", "method": "tools/list", "id": 1}
            )
            
            # Handshake real
            tools_response = await session.list_tools()
            
            # B. LOG VISUAL: Imprimiendo el catálogo de herramientas locales expuestas
            esquema_tools_servidor = {
                "jsonrpc": "2.0",
                "result": {
                    "tools": [
                        {"name": t.name, "description": t.description, "inputSchema": t.inputSchema}
                        for t in tools_response.tools
                    ]
                },
                "id": 1
            }
            mostrar_log_visual_json_rpc(
                direccion="SERVIDOR_RESPONDIENDO",
                metodo="tools/list (RESPONSE)",
                payload=esquema_tools_servidor
            )
            
            # C. Inferencia y ejecución según SDK seleccionado
            if proveedor == "gemini":
                print(f"🧠 [LLM] Consultando decisiones a Google SDK con el modelo '{modelo}'...")
                
                # Convertimos las herramientas de MCP a formato Declarativo de Gemini
                gemini_tools = []
                for tool in tools_response.tools:
                    gemini_tools.append({
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    })
                
                # Instanciar modelo
                model = genai.GenerativeModel(
                    model_name=modelo,
                    tools=gemini_tools
                )
                
                # Generar inferencia (de forma asíncrona)
                response = await model.generate_content_async(prompt)
                
                # D. Procesar Tool Calls de Google Gemini
                partes = response.candidates[0].content.parts
                tool_calls_detected = [p.function_call for p in partes if p.function_call]
                
                if tool_calls_detected:
                    for id_call, fn_call in enumerate(tool_calls_detected):
                        nombre_funcion = fn_call.name
                        argumentos = dict(fn_call.args)
                        
                        # Visualizar llamada de IA
                        mostrar_log_visual_json_rpc(
                            direccion="IA_DECIDIENDO",
                            metodo=f"tool_call (GEMINI SDK) -> {nombre_funcion}",
                            payload={
                                "model": modelo,
                                "action_requested": {
                                    "name": nombre_funcion,
                                    "arguments": argumentos
                                }
                            }
                        )
                        
                        # Visualizar payload de cliente a servidor
                        mostrar_log_visual_json_rpc(
                            direccion="CLIENTE_ENVIANDO",
                            metodo="tools/call (REQUEST)",
                            payload={"jsonrpc": "2.0", "method": "tools/call", "params": {"name": nombre_funcion, "arguments": argumentos}, "id": 20 + id_call}
                        )
                        
                        # Ejecución local real en mcp_server.py
                        resultado = await session.call_tool(nombre_funcion, argumentos)
                        
                        # Visualizar retorno
                        mostrar_log_visual_json_rpc(
                            direccion="SERVIDOR_RESPONDIENDO",
                            metodo="tools/call (RESPONSE)",
                            payload={"jsonrpc": "2.0", "result": {"content": [{"type": "text", "text": resultado.content[0].text}]}, "id": 20 + id_call}
                        )
                    print("🎉 Pipeline MCP orquestado con éxito mediante Google SDK.")
                else:
                    print("\n💬 Respuesta de Gemini:")
                    print(response.text)
                    
            else:
                # FLUJO DE OPENROUTER (OpenAI SDK)
                print(f"🧠 [LLM] Consultando decisiones a OpenRouter con el modelo '{modelo}'...")
                
                openrouter_tools = []
                for tool in tools_response.tools:
                    openrouter_tools.append({
                        "type": "function",
                        "function": {
                            "name": tool.name,
                            "description": tool.description,
                            "parameters": tool.inputSchema
                        }
                    })
                
                messages = [
                    {
                        "role": "system",
                        "content": "Eres un agente científico de datos altamente técnico. Tu tarea es extraer la data de MySQL y transformarla usando tus herramientas locales. Responde de forma muy concisa."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
                
                response = await llm_client.chat.completions.create(
                    model=modelo,
                    messages=messages,
                    tools=openrouter_tools
                )
                
                mensaje_modelo = response.choices[0].message
                
                if mensaje_modelo.tool_calls:
                    for id_call, tool_call in enumerate(mensaje_modelo.tool_calls):
                        nombre_funcion = tool_call.function.name
                        argumentos = json.loads(tool_call.function.arguments)
                        
                        # Visualizar llamada de IA
                        mostrar_log_visual_json_rpc(
                            direccion="IA_DECIDIENDO",
                            metodo=f"tool_call (OPENROUTER) -> {nombre_funcion}",
                            payload={
                                "tool_call_id": tool_call.id,
                                "model": modelo,
                                "action_requested": {
                                    "name": nombre_funcion,
                                    "arguments": argumentos
                                }
                            }
                        )
                        
                        # Visualizar payload
                        mostrar_log_visual_json_rpc(
                            direccion="CLIENTE_ENVIANDO",
                            metodo="tools/call (REQUEST)",
                            payload={"jsonrpc": "2.0", "method": "tools/call", "params": {"name": nombre_funcion, "arguments": argumentos}, "id": 10 + id_call}
                        )
                        
                        # Ejecución local real en mcp_server.py
                        resultado_local = await session.call_tool(nombre_funcion, argumentos)
                        
                        # Visualizar retorno
                        mostrar_log_visual_json_rpc(
                            direccion="SERVIDOR_RESPONDIENDO",
                            metodo="tools/call (RESPONSE)",
                            payload={"jsonrpc": "2.0", "result": {"content": [{"type": "text", "text": resultado_local.content[0].text}]}, "id": 10 + id_call}
                        )
                    print("🎉 Pipeline MCP orquestado con éxito mediante OpenRouter.")
                else:
                    print("\n💬 Respuesta del modelo:")
                    print(mensaje_modelo.content)

print("✅ Orquestador multi-proveedor listo.")

---

## 🚀 Paso 9: Iniciar Inferencia y Orquestación en Vivo
Ejecutamos el flujo final. La celda leerá la IA y transporte elegidos y los procesará.

In [ ]:
await correr_pipeline_mcp_avanzado(prompt_activo, ia_proveedor, modelo_activo)